# BIST Gerçek Veri — Hibrit Test
## Aylık WFO Doğrulaması + Günlük Sinyal Katmanı

**Veri kaynağı:** BIST MCP API (gerçek borsa verisi)
**Aylık katman:** 36 aylık OHLCV → WFO strateji karşılaştırması (look-ahead bias yok)
**Günlük katman:** Son 30 günlük OHLCV → ATR stop, RSI/MACD, Kelly pozisyon boyutu
**Hisseler:** THYAO · GARAN · ASELS · BIMAS · EREGL
**Tarih:** 2026-06-06

In [ ]:
import subprocess, sys
for pkg in ["numpy","pandas","matplotlib","scikit-learn","hmmlearn"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg])

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
try:
    from hmmlearn.hmm import GaussianHMM
    HMM_OK = True
except ImportError:
    HMM_OK = False

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.dpi"] = 110
print("Kütüphaneler hazır. HMM:", HMM_OK)


In [ ]:
PORTFOY_TL      = 100_000
KELLY_FRAKSIYON = 0.25      # quarter-Kelly
MAX_POZISYON    = 0.15      # max %15 portföy
ATR_CARPAN      = 2.0       # 2×ATR stop
RR_HEDEF        = 2.0       # 2:1 risk-reward
WF_TRAIN_MIN    = 18        # aylık min eğitim
WF_TEST         = 3         # aylık test penceresi
WF_STEP         = 3         # kaydırma adımı
SPLIT_ESIK      = -0.40     # bu altı → bölünme şüphesi
HISSELER        = ["THYAO","GARAN","ASELS","BIMAS","EREGL"]


In [ ]:
# ── GERÇEK BIST VERİSİ (MCP API, 2026-06-06) ────────────────────────────────
MONTHLY_DATES = ["2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06", "2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12", "2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06", "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]

MONTHLY_CLOSE = {
    "THYAO": [
        234.3,
        245.0,
        242.5,
        217.3,
        251.0,
        228.6,
        272.75,
        281.75,
        297.75,
        326.0,
        302.75,
        308.75,
        289.25,
        300.5,
        285.0,
        272.5,
        285.0,
        281.0,
        317.0,
        308.25,
        311.0,
        284.25,
        273.25,
        283.5,
        288.0,
        335.0,
        315.0,
        291.5,
        272.75,
        268.5,
        304.0,
        307.5,
        294.25,
        308.25,
        296.75,
        297.0
    ],
    "GARAN": [
        43.9,
        50.2,
        53.25,
        47.0,
        54.05,
        58.35,
        63.5,
        63.1,
        68.5,
        82.45,
        107.8,
        112.4,
        123.8,
        112.3,
        122.4,
        106.9,
        120.5,
        124.4,
        127.3,
        128.7,
        118.0,
        102.7,
        105.9,
        135.0,
        142.8,
        144.1,
        139.8,
        134.6,
        138.9,
        143.5,
        161.3,
        160.1,
        126.5,
        133.8,
        122.9,
        125.0
    ],
    "ASELS": [
        74.6,
        38.52,
        40.8,
        41.34,
        48.84,
        44.98,
        50.6,
        60.0,
        56.1,
        60.75,
        58.65,
        58.6,
        62.5,
        58.4,
        59.25,
        60.85,
        69.45,
        72.5,
        88.7,
        93.45,
        119.9,
        135.0,
        129.6,
        150.8,
        187.0,
        183.3,
        215.0,
        203.6,
        183.4,
        231.7,
        303.25,
        322.0,
        320.25,
        420.25,
        380.25,
        363.0
    ],
    "BIMAS": [
        216.1,
        252.2,
        274.7,
        272.0,
        308.0,
        300.75,
        381.0,
        388.25,
        361.0,
        387.0,
        480.0,
        545.0,
        625.5,
        540.0,
        496.25,
        466.5,
        473.5,
        528.5,
        550.0,
        506.0,
        458.25,
        451.0,
        476.0,
        494.75,
        530.5,
        530.0,
        541.0,
        539.0,
        537.0,
        536.5,
        664.0,
        667.5,
        683.0,
        741.5,
        373.0,
        377.25
    ],
    "EREGL": [
        41.7,
        43.02,
        44.52,
        37.86,
        40.96,
        41.0,
        43.14,
        45.52,
        42.32,
        43.06,
        47.94,
        53.25,
        56.05,
        48.34,
        53.5,
        47.6,
        25.78,
        24.4,
        22.44,
        22.32,
        22.52,
        22.5,
        23.44,
        26.66,
        26.72,
        29.86,
        29.4,
        27.48,
        23.86,
        23.82,
        28.18,
        32.66,
        28.22,
        35.12,
        39.18,
        39.16
    ],
    "XU100": [
        7217,
        7918,
        8335,
        7514,
        7949,
        7470,
        8497,
        9194,
        9142,
        10046,
        10400,
        10648,
        10639,
        9833,
        9666,
        8864,
        9652,
        9831,
        10004,
        9659,
        9659,
        9078,
        9020,
        9949,
        10743,
        11288,
        11012,
        10972,
        10899,
        11262,
        13838,
        13718,
        12791,
        14443,
        13663,
        13694
    ]
}

MONTHLY_HIGH  = {
    "THYAO": [
        237.6,
        269.8,
        258.6,
        255.6,
        269.25,
        264.0,
        277.25,
        294.75,
        299.5,
        327.75,
        332.0,
        317.5,
        318.75,
        308.0,
        309.75,
        287.0,
        300.25,
        305.0,
        317.75,
        327.0,
        339.5,
        325.25,
        309.75,
        293.25,
        299.5,
        346.25,
        340.25,
        318.75,
        298.25,
        283.5,
        308.25,
        352.5,
        299.5,
        335.0,
        316.25,
        301.75
    ],
    "GARAN": [
        45.04,
        58.65,
        54.6,
        55.05,
        55.7,
        64.2,
        69.2,
        69.6,
        72.4,
        84.65,
        110.4,
        118.8,
        138.3,
        131.2,
        130.9,
        125.2,
        122.8,
        135.8,
        139.2,
        132.4,
        145.6,
        119.1,
        114.5,
        135.0,
        145.3,
        154.5,
        153.5,
        147.1,
        140.3,
        145.0,
        162.0,
        169.7,
        154.6,
        147.7,
        139.0,
        130.3
    ],
    "ASELS": [
        77.2,
        81.15,
        43.96,
        44.34,
        49.86,
        50.4,
        51.9,
        67.3,
        60.9,
        62.25,
        64.55,
        62.35,
        66.1,
        64.2,
        63.8,
        62.65,
        71.8,
        74.9,
        90.6,
        94.3,
        125.7,
        141.3,
        158.5,
        151.9,
        187.7,
        188.7,
        222.3,
        223.3,
        206.0,
        237.3,
        339.25,
        324.75,
        359.0,
        434.25,
        450.0,
        409.5
    ],
    "BIMAS": [
        221.2,
        278.0,
        286.2,
        330.4,
        324.5,
        327.0,
        388.75,
        405.0,
        399.75,
        407.75,
        497.75,
        583.5,
        631.0,
        631.0,
        623.0,
        501.0,
        498.5,
        551.5,
        558.0,
        581.0,
        563.5,
        483.5,
        506.0,
        533.0,
        531.5,
        545.0,
        549.0,
        587.0,
        578.0,
        576.0,
        679.5,
        724.5,
        721.0,
        770.5,
        814.5,
        385.5
    ],
    "EREGL": [
        42.0,
        46.18,
        48.48,
        45.58,
        41.96,
        43.56,
        45.6,
        51.0,
        46.94,
        43.88,
        50.15,
        54.45,
        60.25,
        57.85,
        54.05,
        53.25,
        51.85,
        27.86,
        24.94,
        23.84,
        25.86,
        23.32,
        25.18,
        27.0,
        28.1,
        30.38,
        32.7,
        30.1,
        29.48,
        25.04,
        28.72,
        32.72,
        32.28,
        35.94,
        43.72,
        41.22
    ]
}

MONTHLY_LOW   = {
    "THYAO": [
        208.6,
        227.3,
        214.6,
        203.0,
        210.7,
        221.6,
        229.7,
        272.5,
        260.75,
        287.5,
        299.25,
        298.5,
        288.25,
        271.25,
        280.0,
        257.5,
        267.0,
        278.75,
        281.25,
        297.5,
        282.25,
        283.25,
        273.0,
        249.2,
        279.0,
        286.75,
        307.5,
        283.75,
        266.5,
        262.75,
        269.75,
        298.0,
        263.5,
        291.25,
        271.5,
        291.0
    ],
    "GARAN": [
        32.8,
        42.14,
        48.52,
        44.16,
        45.08,
        54.7,
        55.0,
        62.35,
        58.6,
        67.55,
        82.05,
        97.75,
        107.7,
        107.2,
        104.5,
        106.5,
        97.2,
        118.1,
        124.9,
        120.1,
        105.5,
        100.0,
        98.75,
        105.3,
        132.1,
        140.0,
        132.7,
        115.4,
        124.2,
        136.6,
        141.0,
        150.9,
        124.7,
        125.8,
        116.9,
        122.1
    ],
    "ASELS": [
        58.1,
        36.7,
        37.44,
        38.76,
        40.0,
        42.9,
        45.14,
        50.55,
        52.1,
        51.55,
        57.4,
        56.6,
        55.45,
        54.25,
        54.1,
        54.7,
        61.0,
        67.4,
        72.2,
        77.25,
        93.55,
        113.5,
        128.2,
        126.9,
        145.5,
        167.0,
        168.7,
        190.5,
        171.5,
        181.5,
        228.5,
        283.0,
        314.75,
        317.75,
        370.5,
        355.0
    ],
    "BIMAS": [
        179.6,
        213.6,
        252.8,
        271.8,
        268.7,
        295.25,
        289.75,
        366.0,
        342.25,
        353.0,
        388.75,
        475.0,
        536.0,
        531.0,
        491.0,
        445.75,
        443.5,
        468.0,
        500.5,
        504.0,
        399.0,
        420.0,
        452.0,
        451.5,
        483.0,
        508.0,
        483.25,
        518.5,
        523.0,
        512.0,
        535.5,
        638.0,
        607.5,
        681.0,
        371.75,
        369.75
    ],
    "EREGL": [
        36.82,
        39.3,
        40.28,
        37.62,
        36.3,
        37.76,
        40.92,
        43.22,
        41.52,
        38.22,
        43.28,
        46.08,
        50.8,
        46.7,
        45.0,
        45.5,
        24.62,
        24.4,
        22.38,
        21.48,
        20.32,
        20.1,
        21.98,
        22.92,
        26.22,
        26.72,
        25.42,
        25.84,
        23.74,
        23.46,
        23.88,
        26.84,
        26.78,
        27.94,
        34.4,
        38.78
    ]
}

MONTHLY_VOL   = {
    "THYAO": [
        712829789,
        955781274,
        844413045,
        847733057,
        945549348,
        656086730,
        738355873,
        876090619,
        716801095,
        752598784,
        728979765,
        529404708,
        651093294,
        596800829,
        536196266,
        489339098,
        608885136,
        565585689,
        661332376,
        677961675,
        783905405,
        511317240,
        523673871,
        525777934,
        627577550,
        661061375,
        799463382,
        845629973,
        715932594,
        695549148,
        1220944447,
        1336430116,
        867805812,
        1048193772,
        706351958,
        193842895
    ],
    "GARAN": [
        1295635322,
        1629750096,
        1174397854,
        823188674,
        702689684,
        735343827,
        813376444,
        717503241,
        705368910,
        454243565,
        543436250,
        298080766,
        509268522,
        430614252,
        452828653,
        484039458,
        580926814,
        417141906,
        550407531,
        520989859,
        711871375,
        522545953,
        483377379,
        560464368,
        595904576,
        408851117,
        751984364,
        853448868,
        625434860,
        680536836,
        891926941,
        744961712,
        578980725,
        613780737,
        543638363,
        146480095
    ],
    "ASELS": [
        1422891729,
        1441490298,
        1754165932,
        1772423308,
        1185222993,
        949302633,
        1131949093,
        1335227868,
        990706049,
        740671951,
        931496626,
        588335254,
        831165449,
        704148397,
        628212744,
        729297685,
        758461160,
        577885478,
        649293387,
        820407946,
        1370178866,
        753441067,
        755463898,
        662613270,
        715119159,
        471535618,
        727950496,
        628197585,
        565557635,
        642024824,
        1198210103,
        903676325,
        631877266,
        585960618,
        313177360,
        158263952
    ],
    "BIMAS": [
        98660760,
        182517223,
        118550614,
        160231851,
        84463411,
        75945292,
        76016829,
        70724740,
        80071713,
        67082596,
        75767118,
        65730392,
        77565318,
        60719205,
        86988029,
        124983426,
        106518620,
        79247932,
        119296942,
        125199965,
        244375372,
        120018448,
        108548088,
        98735550,
        104311991,
        82308232,
        118208058,
        129022858,
        118227921,
        152857244,
        172293888,
        143363779,
        110902092,
        93573221,
        114652075,
        50245883
    ],
    "EREGL": [
        2045888856,
        3630058007,
        2601184595,
        2156082740,
        1513549678,
        1449878425,
        1840177097,
        3921708731,
        2333257384,
        1387652768,
        3046694614,
        2071473858,
        3469247660,
        2186614295,
        1762893641,
        1950535140,
        1801123690,
        3377829045,
        3664808389,
        4043526979,
        4430662359,
        2808233039,
        3169402675,
        3820357976,
        3196244161,
        3875492792,
        4965318376,
        4091342675,
        4434234328,
        3641031437,
        3736953941,
        4502335795,
        2331632926,
        4166245953,
        3007971383,
        887058122
    ]
}

DAILY = {
    "THYAO": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "open": [
            318,
            321.5,
            329,
            329.5,
            323,
            324.5,
            319.5,
            316.75,
            310,
            308.5,
            300.75,
            307,
            311.75,
            310,
            308.25,
            308,
            307.75,
            305.25,
            303.5,
            299.5,
            292,
            295,
            274,
            299,
            297.5,
            298.5,
            294.5,
            297.25,
            299,
            299.75
        ],
        "high": [
            335,
            329.25,
            334.5,
            331.25,
            326.25,
            326.75,
            320.25,
            317.75,
            311.5,
            310.75,
            304.25,
            316.25,
            315.25,
            314.75,
            309.5,
            310,
            310,
            308.5,
            304.25,
            301,
            300.25,
            295.75,
            290.25,
            300.5,
            298,
            300.5,
            300.25,
            301.75,
            301.5,
            300.25
        ],
        "low": [
            317.75,
            321.25,
            327,
            319.5,
            319,
            320.25,
            315.25,
            313,
            305.25,
            300.5,
            298.25,
            304.75,
            311.25,
            306.75,
            306.25,
            305.25,
            304,
            305.25,
            298.5,
            294.5,
            289.75,
            273.25,
            271.5,
            296,
            295.25,
            291,
            294.25,
            296.5,
            296.5,
            295.25
        ],
        "close": [
            329,
            328.25,
            327,
            323.5,
            325,
            320.5,
            315.75,
            314.5,
            308.25,
            300.75,
            301,
            309.75,
            312.5,
            311,
            308.5,
            305.75,
            304,
            305.75,
            300,
            294.5,
            295,
            274,
            288,
            297.5,
            296.75,
            291.5,
            300,
            297,
            299.75,
            297
        ],
        "volume": [
            108841541,
            62881814,
            41012885,
            37829627,
            37892432,
            39753416,
            27040131,
            30230456,
            55904761,
            57028327,
            39119327,
            102928965,
            54994125,
            52591635,
            32165704,
            25842992,
            31057355,
            33906131,
            29909261,
            36354194,
            34515033,
            57504394,
            62579877,
            44110081,
            11744557,
            43696715,
            42876580,
            36533990,
            45830613,
            24904997
        ]
    },
    "GARAN": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "open": [
            137.1,
            144,
            144.9,
            143.1,
            138.8,
            138.1,
            136.5,
            135.8,
            132,
            134.2,
            129.6,
            131.8,
            138,
            136.7,
            136.6,
            137.7,
            134.2,
            132.1,
            131.5,
            128.7,
            128.8,
            129,
            120.5,
            123,
            125.1,
            123.7,
            124.3,
            128.8,
            126.7,
            130
        ],
        "high": [
            147.1,
            147.7,
            145.8,
            144,
            139.9,
            139.4,
            137.1,
            136.4,
            134.8,
            134.9,
            130.8,
            137.5,
            139,
            138.2,
            138.6,
            138.3,
            134.6,
            134.2,
            131.7,
            130.7,
            131.1,
            130.3,
            123.5,
            125.6,
            125.9,
            126.6,
            130,
            129.4,
            129.8,
            130.3
        ],
        "low": [
            136.4,
            142.8,
            141.8,
            139.3,
            136.4,
            136.2,
            134.3,
            132.3,
            131,
            128.9,
            128.3,
            130.9,
            136.6,
            134.8,
            136.1,
            133.3,
            130.7,
            131,
            129,
            126.6,
            127.6,
            120,
            116.9,
            121.7,
            122.9,
            122.1,
            124.2,
            125.8,
            125.1,
            124.9
        ],
        "close": [
            146.4,
            144,
            141.8,
            139.4,
            138,
            136.7,
            134.6,
            132.7,
            133.8,
            129.6,
            129.4,
            136.6,
            137.6,
            137.3,
            138.2,
            133.4,
            131.5,
            133.7,
            129.8,
            129.3,
            129.8,
            120,
            122.3,
            125.2,
            122.9,
            123.1,
            128.8,
            125.9,
            129.5,
            125
        ],
        "volume": [
            49171307,
            37968647,
            20234879,
            22544707,
            22759476,
            23883931,
            18070338,
            24681843,
            27400523,
            30032824,
            23142970,
            74603470,
            47414602,
            34280123,
            38541286,
            28038200,
            28761121,
            24762552,
            26965070,
            26936943,
            24242460,
            27281106,
            59917827,
            38385587,
            10332222,
            29134069,
            27336972,
            28021323,
            34663394,
            27324337
        ]
    },
    "ASELS": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "open": [
            412.5,
            418,
            409,
            399,
            399,
            393.75,
            420.75,
            415.5,
            427.25,
            422,
            432.75,
            430.75,
            437,
            431,
            430,
            433,
            420,
            414.75,
            415.5,
            416.75,
            402,
            397.75,
            370.5,
            411.25,
            399.75,
            385,
            385,
            409.5,
            388.5,
            362.75
        ],
        "high": [
            417.5,
            421.5,
            412.25,
            406.75,
            404.5,
            416.25,
            424.75,
            434.25,
            431,
            443.5,
            434.5,
            450,
            437,
            434.5,
            438.75,
            433.25,
            423.5,
            421,
            419,
            419,
            403.25,
            409.25,
            412,
            413,
            401.75,
            398.25,
            408.5,
            409.5,
            393.5,
            367.75
        ],
        "low": [
            407,
            408,
            396.5,
            391,
            390.25,
            391,
            414.25,
            415.5,
            418.25,
            421.5,
            424.25,
            429.5,
            423,
            426.25,
            425,
            419.25,
            410.75,
            413.75,
            410.5,
            402,
            389.25,
            377.75,
            370.5,
            396,
            380.25,
            383,
            383,
            381.25,
            356.75,
            355
        ],
        "close": [
            414,
            408,
            396.5,
            396.5,
            392,
            415,
            414.5,
            425.5,
            420.25,
            431.75,
            427.25,
            434,
            428,
            428.5,
            431.75,
            419.25,
            412.75,
            415.5,
            415,
            402,
            393.5,
            377.75,
            410,
            401.75,
            380.25,
            384,
            407.75,
            383.5,
            362.25,
            363
        ],
        "volume": [
            20981008,
            17267189,
            13215889,
            23228306,
            18064062,
            29311856,
            28470534,
            38264448,
            27047316,
            35636201,
            15656683,
            30033225,
            17111142,
            17908234,
            21023942,
            13735481,
            13698866,
            10379851,
            15045515,
            14506796,
            21017807,
            20097708,
            28358776,
            15337705,
            23629428,
            28628121,
            27766669,
            33258536,
            44453259,
            24157367
        ]
    },
    "BIMAS": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "open": [
            748,
            755.5,
            758.5,
            758,
            761,
            761.5,
            752,
            732.5,
            723,
            744,
            731,
            756,
            782,
            786,
            791,
            805,
            786.5,
            411.5,
            413,
            402.75,
            391,
            392.75,
            372.5,
            393,
            380,
            373.75,
            374,
            379.5,
            382.5,
            373.75
        ],
        "high": [
            770.5,
            764.5,
            765,
            770.5,
            761.5,
            764,
            752.5,
            739,
            743.5,
            745.5,
            751,
            785,
            790,
            802.5,
            792,
            811.5,
            814.5,
            425,
            413,
            403,
            399.25,
            399.75,
            395.5,
            397.5,
            380.75,
            379.25,
            382,
            385.25,
            385.5,
            379.75
        ],
        "low": [
            742.5,
            750,
            752.5,
            752.5,
            747,
            751.5,
            729,
            721.5,
            720.5,
            725.5,
            728.5,
            755,
            778.5,
            777,
            771.5,
            775.5,
            786.5,
            409.25,
            398,
            388,
            385.5,
            375,
            372,
            375.75,
            371.75,
            369.75,
            370.25,
            374.25,
            370,
            371
        ],
        "close": [
            765,
            756,
            752.5,
            763,
            760,
            752.5,
            730,
            727,
            741.5,
            730,
            747,
            781,
            784,
            790,
            774.5,
            781,
            813,
            414,
            405.25,
            392.5,
            392.75,
            376.5,
            392.75,
            378.75,
            373,
            371.25,
            380.75,
            381,
            373,
            377.25
        ],
        "volume": [
            5990677,
            3426728,
            2912672,
            4334906,
            3729933,
            3008305,
            3480818,
            2956005,
            5286809,
            4234278,
            3634808,
            7580243,
            3323695,
            3947561,
            4457296,
            7070464,
            7813413,
            8115738,
            6634672,
            8402236,
            10839321,
            8847731,
            12459095,
            10897046,
            6394478,
            10572396,
            11495125,
            10514304,
            9543504,
            8120554
        ]
    },
    "EREGL": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "open": [
            30.96,
            31.7,
            33.52,
            34,
            32.52,
            33.22,
            33.4,
            33.24,
            33.64,
            35.36,
            35.96,
            37.8,
            38,
            38.42,
            41.8,
            40.9,
            40.5,
            40.88,
            39.48,
            39.44,
            38.46,
            38.64,
            36,
            39.12,
            39.74,
            39.18,
            40.46,
            40.54,
            40.58,
            40.08
        ],
        "high": [
            32.42,
            33.7,
            34.48,
            34.04,
            33.06,
            34.14,
            33.84,
            34.32,
            35.94,
            36.08,
            37.96,
            38.64,
            39.02,
            41.3,
            43.72,
            41.4,
            40.9,
            41.1,
            39.86,
            39.66,
            39.2,
            38.72,
            38.68,
            40.7,
            39.88,
            40.18,
            41.12,
            40.86,
            41.22,
            40.54
        ],
        "low": [
            30.92,
            31.7,
            33.48,
            32.04,
            32.2,
            33.12,
            32.76,
            33.24,
            33.64,
            34.4,
            35.9,
            36.94,
            37.96,
            38.3,
            40.58,
            40,
            39.44,
            39.3,
            39.04,
            38.22,
            37.94,
            35.02,
            35.48,
            38.9,
            39.18,
            39.18,
            40.32,
            40.18,
            38.78,
            38.88
        ],
        "close": [
            32.04,
            33.3,
            33.72,
            32.6,
            33,
            33.6,
            32.98,
            33.72,
            35.12,
            35.82,
            37.22,
            37.68,
            38.48,
            41.26,
            40.86,
            40.08,
            40.5,
            39.74,
            39.62,
            38.54,
            38.64,
            35.18,
            38.68,
            39.48,
            39.18,
            40,
            41,
            40.3,
            40.1,
            39.16
        ],
        "volume": [
            214957950,
            235690620,
            173341343,
            157118559,
            120935169,
            139796883,
            113229703,
            266432560,
            223929844,
            221260604,
            234340867,
            306892390,
            155741673,
            270521370,
            334772810,
            229566550,
            183857127,
            134596775,
            152182700,
            135047714,
            85624045,
            123679155,
            210744663,
            187516071,
            41626869,
            235926121,
            200428550,
            114552214,
            177175503,
            158975734
        ]
    },
    "XU100": {
        "dates": [
            "2026-04-17",
            "2026-04-20",
            "2026-04-21",
            "2026-04-22",
            "2026-04-24",
            "2026-04-27",
            "2026-04-28",
            "2026-04-29",
            "2026-04-30",
            "2026-05-04",
            "2026-05-05",
            "2026-05-06",
            "2026-05-07",
            "2026-05-08",
            "2026-05-11",
            "2026-05-12",
            "2026-05-13",
            "2026-05-14",
            "2026-05-15",
            "2026-05-18",
            "2026-05-20",
            "2026-05-21",
            "2026-05-22",
            "2026-05-25",
            "2026-05-26",
            "2026-06-01",
            "2026-06-02",
            "2026-06-03",
            "2026-06-04",
            "2026-06-05"
        ],
        "close": [
            14587.93,
            14484.91,
            14375.4,
            14335.49,
            14409.07,
            14594.01,
            14329.34,
            14311.19,
            14442.56,
            14369.61,
            14495.77,
            14917.43,
            15040.25,
            15062.65,
            15133.54,
            14779.93,
            14598.47,
            14644.7,
            14367.6,
            14029.54,
            14012.01,
            13163.88,
            13808.2,
            13890.91,
            13662.75,
            13703.96,
            14200.2,
            13965.65,
            13872.25,
            13694.19
        ]
    }
}

print("Aylık veri:", len(MONTHLY_DATES), "bar")
print("Günlük veri:", {k: len(v['close']) for k,v in DAILY.items() if 'close' in v})


In [ ]:
def build_monthly_df(ticker):
    """Monthly OHLCV → DataFrame with returns and split detection."""
    dates = pd.to_datetime(MONTHLY_DATES)
    close = pd.Series(MONTHLY_CLOSE[ticker], index=dates, name="close")
    high  = pd.Series(MONTHLY_HIGH.get(ticker, [np.nan]*len(dates)), index=dates, name="high")
    low   = pd.Series(MONTHLY_LOW.get(ticker, [np.nan]*len(dates)), index=dates, name="low")
    vol   = pd.Series(MONTHLY_VOL.get(ticker, [np.nan]*len(dates)), index=dates, name="volume")
    xu100 = pd.Series(MONTHLY_CLOSE["XU100"], index=dates, name="xu100")

    df = pd.DataFrame({"close": close, "high": high, "low": low, "volume": vol, "xu100": xu100})

    # Log returns
    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    # Detect splits: monthly return < -40%
    df["is_split"] = df["log_ret"] < SPLIT_ESIK
    df.loc[df["is_split"], "log_ret"] = np.nan   # NaN o ayı

    # Features
    df["sma3"]     = df["close"].rolling(3).mean()
    df["sma6"]     = df["close"].rolling(6).mean()
    df["ret_3m"]   = df["close"].pct_change(3)
    df["ret_6m"]   = df["close"].pct_change(6)
    df["vol_6m"]   = df["log_ret"].rolling(6).std()
    df["shr_6m"]   = (df["log_ret"].rolling(6).mean()
                      / (df["log_ret"].rolling(6).std() + 1e-9))
    df["vol_ratio"]= (df["volume"] / df["volume"].rolling(6).mean()).fillna(1)
    # Relative strength vs XU100 (3m)
    log_xu = np.log(df["xu100"] / df["xu100"].shift(3))
    log_st = np.log(df["close"] / df["close"].shift(3))
    df["rel_xu100"] = (log_st - log_xu).fillna(0)
    return df

# Build all monthly DataFrames
MDF = {t: build_monthly_df(t) for t in HISSELER}
splits_found = {t: MDF[t]["is_split"].sum() for t in HISSELER}
print("Aylık DataFrame hazır. Split tespiti:", splits_found)


In [ ]:
class MACTStrategy:
    """MA Crossover: long when price > SMA3 & SMA6."""
    name = "MACT"
    def fit(self, df): pass
    def predict(self, df):
        sig = ((df["close"] > df["sma3"]) & (df["close"] > df["sma6"])).astype(float)
        return sig.shift(1).fillna(0)  # 1-ay gecikme

class MOMStrategy:
    """3-Aylık Momentum: positive = long."""
    name = "MOM3"
    def fit(self, df): pass
    def predict(self, df):
        sig = (df["ret_3m"] > 0).astype(float)
        return sig.shift(1).fillna(0)

class RAMStrategy:
    """Risk-Adjusted Momentum: 6-aylık Sharpe > eşik."""
    name = "RAM"
    def __init__(self): self.threshold = 0.15
    def fit(self, df):
        best, best_s = 0.15, -np.inf
        for t in [0, 0.10, 0.15, 0.20, 0.30]:
            sig = (df["shr_6m"] > t).astype(float).shift(1).fillna(0)
            r   = (sig * df["log_ret"]).dropna()
            if len(r) > 3 and r.std() > 0:
                s = r.mean() / r.std() * np.sqrt(12)
                if s > best_s: best_s, best = s, t
        self.threshold = best
    def predict(self, df):
        sig = (df["shr_6m"] > self.threshold).astype(float)
        return sig.shift(1).fillna(0)

class HMMStrategy:
    """HMM Rejim Algılama (2 durum: Boğa/Ayı)."""
    name = "HMM"
    def __init__(self): self.model, self.bull = None, 0
    def fit(self, df):
        if not HMM_OK: return
        feats = df[["log_ret", "rel_xu100"]].dropna()
        if len(feats) < 12:
            self.model = None; return
        try:
            m = GaussianHMM(n_components=2, covariance_type="full",
                            n_iter=200, random_state=42)
            m.fit(StandardScaler().fit_transform(feats.values))
            self.bull  = int(np.argmax(m.means_[:, 0]))
            self.model = m
            self._scaler = StandardScaler().fit(feats.values)
            self._cols   = feats.columns.tolist()
        except Exception:
            self.model = None
    def predict(self, df):
        if self.model is None:
            return pd.Series(0.0, index=df.index)
        feats = df[self._cols].dropna()
        try:
            X    = self._scaler.transform(feats.values)
            sts  = self.model.predict(X)
            sig  = pd.Series(0.0, index=df.index)
            sig.loc[feats.index] = (sts == self.bull).astype(float)
            return sig.shift(1).fillna(0)
        except Exception:
            return pd.Series(0.0, index=df.index)

class EnsembleStrategy:
    name = "ENSE"
    def __init__(self): self.subs = [MACTStrategy(), MOMStrategy(), RAMStrategy(), HMMStrategy()]
    def fit(self, df):
        for s in self.subs: s.fit(df)
    def predict(self, df):
        votes = pd.concat([s.predict(df) for s in self.subs], axis=1)
        return (votes.mean(axis=1) >= 0.5).astype(float)

print("Strateji sınıfları hazır: MACT, MOM3, RAM, HMM, ENSE")


In [ ]:
def walk_forward(df, strategies, train_min=WF_TRAIN_MIN, test_size=WF_TEST, step=WF_STEP):
    """Expanding-window WFO. Returns {name: return_series}."""
    n   = len(df)
    ret = df["log_ret"]
    out = {s.name: pd.Series(np.nan, index=df.index) for s in strategies}

    fold_count = 0
    for fold_start in range(train_min, n - test_size + 1, step):
        train_df = df.iloc[:fold_start].copy()
        # Fit on training data
        for s in strategies:
            s.fit(train_df)
        # Predict on full history up to fold_start+test_size (signals are lagged internally)
        full_to_test = df.iloc[:fold_start + test_size].copy()
        for s in strategies:
            sig = s.predict(full_to_test)
            test_idx = df.index[fold_start:fold_start + test_size]
            test_ret = ret.iloc[fold_start:fold_start + test_size]
            # Only store test-period returns
            out[s.name].iloc[fold_start:fold_start + test_size] = (
                sig.reindex(test_idx).fillna(0).values * test_ret.fillna(0).values
            )
        fold_count += 1

    return out, fold_count

def calc_metrics(ret_series, annual_factor=12):
    r = ret_series.dropna()
    if len(r) == 0:
        return dict(total=0, sharpe=0, maxdd=0, winrate=0, n_months=0)
    cum = (1 + r).cumprod()
    dd  = (cum / cum.cummax() - 1).min()
    sr  = (r.mean() / (r.std() + 1e-9)) * np.sqrt(annual_factor)
    wr  = (r > 0).mean()
    tot = cum.iloc[-1] - 1
    return dict(total=round(tot*100,1), sharpe=round(sr,3),
                maxdd=round(dd*100,1), winrate=round(wr*100,1), n_months=len(r))

print("WFO motoru hazır.")


In [ ]:
STRATEGIES = [MACTStrategy(), MOMStrategy(), RAMStrategy(), HMMStrategy(), EnsembleStrategy()]
WFO_RESULTS  = {}   # {ticker: {strat_name: ret_series}}
WFO_METRICS  = {}   # {ticker: {strat_name: metrics_dict}}
WFO_CHAMP    = {}   # {ticker: best_strat_name}

print("WFO çalışıyor...")
for ticker in HISSELER:
    df = MDF[ticker]
    strat_instances = [MACTStrategy(), MOMStrategy(), RAMStrategy(), HMMStrategy(), EnsembleStrategy()]
    ret_dict, folds = walk_forward(df, strat_instances)
    WFO_RESULTS[ticker] = ret_dict
    WFO_METRICS[ticker] = {k: calc_metrics(v) for k, v in ret_dict.items()}
    # Buy-and-hold reference
    bh_ret = df["log_ret"].iloc[WF_TRAIN_MIN:]
    WFO_METRICS[ticker]["BuyHold"] = calc_metrics(bh_ret)
    # Champion: highest Sharpe among WFO strategies
    best = max(WFO_METRICS[ticker].items(),
               key=lambda x: x[1]["sharpe"] if x[0] != "BuyHold" else -999)
    WFO_CHAMP[ticker] = best[0]
    print(f"  {ticker}: {folds} fold | Şampiyon → {best[0]} "
          f"Sharpe={best[1]['sharpe']:.3f}  Getiri={best[1]['total']:.1f}%")


In [ ]:
rows = []
for ticker in HISSELER:
    for strat, m in WFO_METRICS[ticker].items():
        rows.append(dict(Hisse=ticker, Strateji=strat,
                         Getiri_pct=m["total"], Sharpe=m["sharpe"],
                         MaxDD_pct=m["maxdd"], WinRate_pct=m["winrate"],
                         N_Ay=m["n_months"]))

df_table = pd.DataFrame(rows).sort_values(["Hisse","Sharpe"], ascending=[True,False])
df_table["Şampiyon"] = df_table.apply(
    lambda r: "★" if WFO_CHAMP.get(r["Hisse"]) == r["Strateji"] else "", axis=1)

print("\n═══════════════════════════════════════════════════════════════════════")
print("  GERÇEK VERİ WFO SONUÇLARI — Aylık Strateji Karşılaştırması")
print("═══════════════════════════════════════════════════════════════════════")
print(df_table.to_string(index=False))
print("═══════════════════════════════════════════════════════════════════════")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("BIST Gerçek Veri — WFO Aylık Kümülatif Getiri", fontsize=14, fontweight="bold")
axes_flat = axes.flatten()

colors = {"MACT":"#2196F3","MOM3":"#4CAF50","RAM":"#FF9800","HMM":"#9C27B0","ENSE":"#F44336","BuyHold":"#607D8B"}

for i, ticker in enumerate(HISSELER):
    ax = axes_flat[i]
    for strat, ret_series in WFO_RESULTS[ticker].items():
        r = ret_series.dropna()
        if len(r) == 0: continue
        cum = (1 + r).cumprod()
        lw  = 2.5 if strat == WFO_CHAMP[ticker] else 1.2
        ls  = "-" if strat == WFO_CHAMP[ticker] else "--"
        ax.plot(cum.index, cum.values, label=f"{strat} ({WFO_METRICS[ticker][strat]['total']:.0f}%)",
                color=colors.get(strat,"gray"), lw=lw, ls=ls)
    # Buy-Hold
    bh = MDF[ticker]["log_ret"].iloc[WF_TRAIN_MIN:].dropna()
    bh_cum = (1 + bh).cumprod()
    ax.plot(bh_cum.index, bh_cum.values, label=f"B&H ({WFO_METRICS[ticker]['BuyHold']['total']:.0f}%)",
            color=colors["BuyHold"], lw=1, ls=":", alpha=0.7)
    ax.axhline(1, color="white", lw=0.5, alpha=0.3)
    ax.set_title(f"{ticker}  [Şampiyon: {WFO_CHAMP[ticker]}]", fontweight="bold")
    ax.legend(fontsize=7, loc="upper left")
    ax.set_ylabel("Kümülatif")
    ax.xaxis.set_tick_params(rotation=30)

# 6th panel: Sharpe comparison heatmap
ax6 = axes_flat[5]
strats_order = ["MACT","MOM3","RAM","HMM","ENSE","BuyHold"]
heat_data = []
for ticker in HISSELER:
    heat_data.append([WFO_METRICS[ticker].get(s,{}).get("sharpe",np.nan) for s in strats_order])
heat_arr = np.array(heat_data)
im = ax6.imshow(heat_arr, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=2)
ax6.set_xticks(range(len(strats_order))); ax6.set_xticklabels(strats_order, rotation=30, fontsize=8)
ax6.set_yticks(range(len(HISSELER))); ax6.set_yticklabels(HISSELER)
ax6.set_title("Sharpe Isı Haritası", fontweight="bold")
for r in range(len(HISSELER)):
    for c in range(len(strats_order)):
        v = heat_arr[r,c]
        if not np.isnan(v):
            ax6.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8,
                     color="black" if abs(v)<1 else "white")
plt.colorbar(im, ax=ax6)
plt.tight_layout()
plt.savefig("/tmp/wfo_gercek_veri.png", dpi=120, bbox_inches="tight")
plt.show()
print("Grafik: /tmp/wfo_gercek_veri.png")


In [ ]:
def build_daily_df(ticker):
    d = DAILY[ticker]
    n = len(d["dates"])
    df = pd.DataFrame({
        "date":   pd.to_datetime(d["dates"]),
        "open":   d.get("open",   d["close"]),
        "high":   d.get("high",   d["close"]),
        "low":    d.get("low",    d["close"]),
        "close":  d["close"],
        "volume": d.get("volume", [0]*n)
    }).set_index("date")
    # ATR(14)
    prev_close = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - prev_close).abs(),
        (df["low"]  - prev_close).abs()
    ], axis=1).max(axis=1)
    df["ATR14"] = tr.ewm(span=14, adjust=False).mean()
    # RSI(14)
    delta  = df["close"].diff()
    gain   = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss   = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    df["RSI14"] = 100 - 100 / (1 + gain / (loss + 1e-9))
    # MACD(12,26,9)
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()
    df["MACD"]  = ema12 - ema26
    df["MACDs"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACDh"] = df["MACD"] - df["MACDs"]
    # Bollinger(20,2)
    df["BB_mid"] = df["close"].rolling(20).mean()
    df["BB_std"] = df["close"].rolling(20).std()
    df["BB_z"]   = (df["close"] - df["BB_mid"]) / (df["BB_std"] + 1e-9)
    # Volume ratio
    df["vol_ratio"] = df["volume"] / (df["volume"].rolling(10).mean() + 1e-9)
    return df

DAILY_DF = {t: build_daily_df(t) for t in HISSELER}
DAILY_DF["XU100"] = build_daily_df("XU100")

# Add XU100 close to each stock df
xu_close = pd.Series(DAILY["XU100"]["close"],
                     index=pd.to_datetime(DAILY["XU100"]["dates"]))
for t in HISSELER:
    DAILY_DF[t]["xu100"] = xu_close.reindex(DAILY_DF[t].index).ffill()

print("Günlük DataFrame hazır.")
for t in HISSELER:
    d = DAILY_DF[t].iloc[-1]
    print(f"  {t}: Son={d['close']:.2f} TL  ATR={d['ATR14']:.2f}  RSI={d['RSI14']:.1f}  MACD_hist={d['MACDh']:+.3f}")


In [ ]:
def calc_kelly(ticker):
    """Quarter-Kelly from WFO monthly returns of champion strategy."""
    champ = WFO_CHAMP[ticker]
    ret   = WFO_RESULTS[ticker][champ].dropna()
    active = ret[ret != 0]
    if len(active) < 4:
        return dict(kelly=0.05, win_rate=0.5, payoff=1.0, edge=0.0)
    wins   = active[active > 0]
    losses = active[active < 0]
    if len(losses) == 0:
        return dict(kelly=MAX_POZISYON, win_rate=1.0, payoff=99.0, edge=1.0)
    p = len(wins) / len(active)
    b = wins.mean() / abs(losses.mean() + 1e-9)
    raw = max((p * b - (1 - p)) / (b + 1e-9), 0)
    kelly = min(raw * KELLY_FRAKSIYON, MAX_POZISYON)
    return dict(kelly=round(kelly,4), win_rate=round(p,3),
                payoff=round(b,3), edge=round(p*b-(1-p),4))

def calc_atr_stop(ticker):
    """2×ATR stop-loss and 2:1 R:R target from daily data."""
    d     = DAILY_DF[ticker].iloc[-1]
    entry = d["close"]
    atr   = d["ATR14"]
    stop  = entry - ATR_CARPAN * atr
    target= entry + RR_HEDEF * ATR_CARPAN * atr
    return dict(entry=round(entry,2), stop=round(stop,2), target=round(target,2),
                atr=round(atr,2), stop_pct=round((stop/entry-1)*100,1),
                target_pct=round((target/entry-1)*100,1))

KELLY_INFO = {t: calc_kelly(t) for t in HISSELER}
ATR_INFO   = {t: calc_atr_stop(t) for t in HISSELER}
print("Kelly + ATR hesaplandı:")
for t in HISSELER:
    k = KELLY_INFO[t]; a = ATR_INFO[t]
    print(f"  {t}: Kelly={k['kelly']*100:.1f}%  WR={k['win_rate']*100:.0f}%  "
          f"Payoff={k['payoff']:.2f}x  Stop={a['stop']:.2f} ({a['stop_pct']:.1f}%)  "
          f"Hedef={a['target']:.2f} ({a['target_pct']:.1f}%)")


In [ ]:
def monthly_signal_today(ticker):
    """Apply champion strategy logic to latest monthly bar."""
    df = MDF[ticker]
    champ = WFO_CHAMP[ticker]
    last = df.iloc[-1]
    if champ == "MACT":
        sig = 1 if (last["close"] > last["sma3"] and last["close"] > last["sma6"]) else 0
    elif champ == "MOM3":
        sig = 1 if last["ret_3m"] > 0 else 0
    elif champ == "RAM":
        t = RAMStrategy(); t.fit(df.iloc[:-1])
        sig = 1 if last["shr_6m"] > t.threshold else 0
    elif champ == "HMM":
        h = HMMStrategy(); h.fit(df.iloc[-24:])
        s = h.predict(df.iloc[-6:])
        sig = int(s.iloc[-1]) if len(s) > 0 else 0
    else:  # ENSE
        votes = []
        for Cls in [MACTStrategy, MOMStrategy, RAMStrategy, HMMStrategy]:
            s = Cls()
            if hasattr(s, "fit"): s.fit(df.iloc[:-1])
            p = s.predict(df.iloc[-6:])
            votes.append(int(p.iloc[-1]) if len(p)>0 else 0)
        sig = 1 if sum(votes) >= 2 else 0
    return sig

def daily_signal_today(ticker):
    """Daily RSI + MACD composite signal."""
    d = DAILY_DF[ticker].iloc[-1]
    rsi_sig  = 1 if d["RSI14"] < 70 and d["RSI14"] > 40 else 0
    macd_sig = 1 if d["MACDh"] > 0 else 0
    bb_sig   = 1 if d["BB_z"] < 1.5 else 0   # not overbought
    return rsi_sig + macd_sig + bb_sig  # 0-3 composite

SEP = "═"*68
print(f"\n{SEP}")
print("  BUGÜNKÜ SİNYAL RAPORU  —  2026-06-06")
print(SEP)

all_signals = []
for ticker in HISSELER:
    mon_sig = monthly_signal_today(ticker)   # 0/1
    day_score = daily_signal_today(ticker)   # 0-3
    final_sig = "AL 🔑" if (mon_sig == 1 and day_score >= 2) else                 "BEKLE" if mon_sig == 1 else "FLAT"

    k = KELLY_INFO[ticker]
    a = ATR_INFO[ticker]
    pozisyon_tl  = round(PORTFOY_TL * k["kelly"], 0)
    lot_count    = int(pozisyon_tl // a["entry"]) if a["entry"] > 0 else 0
    risk_tl      = round(lot_count * (a["entry"] - a["stop"]), 0)

    print(f"\n  {'─'*64}")
    print(f"  {ticker}  →  {final_sig}  (Aylık şampiyon: {WFO_CHAMP[ticker]})")
    print(f"  {'─'*64}")
    print(f"  WFO Toplam Getiri : {WFO_METRICS[ticker][WFO_CHAMP[ticker]]['total']:+.1f}%  "
          f"Sharpe: {WFO_METRICS[ticker][WFO_CHAMP[ticker]]['sharpe']:.3f}")
    print(f"  Kelly Pozisyon    : %{k['kelly']*100:.1f}  "
          f"(Ham Kelly %{k['kelly']/KELLY_FRAKSIYON*100:.1f}  WR={k['win_rate']*100:.0f}%  Payoff={k['payoff']:.2f}x)")
    print(f"  Portföy TL        : {PORTFOY_TL:,} TL  →  Pozisyon: {pozisyon_tl:,.0f} TL")
    print(f"  Tahmini Lot       : {lot_count} adet")
    print(f"  GİRİŞ             : {a['entry']:.2f} TL")
    print(f"  STOP-LOSS (2×ATR) : {a['stop']:.2f} TL  ({a['stop_pct']:.1f}%)")
    print(f"  HEDEF (2:1 R:R)   : {a['target']:.2f} TL  ({a['target_pct']:.1f}%)")
    print(f"  Riske Edilen TL   : {risk_tl:,.0f} TL")
    d = DAILY_DF[ticker].iloc[-1]
    print(f"  ─── Günlük Teknik Göstergeler ───────────────────────────────")
    print(f"  RSI(14)={d['RSI14']:.1f}  MACD_hist={d['MACDh']:+.3f}  BB_z={d['BB_z']:+.2f}  "
          f"ATR={d['ATR14']:.2f}  VolRatio={d['vol_ratio']:.2f}x")

    all_signals.append(dict(ticker=ticker, sinyal=final_sig, kelly_pct=k["kelly"]*100,
                            pozisyon_tl=pozisyon_tl, stop=a["stop"], hedef=a["target"],
                            wfo_getiri=WFO_METRICS[ticker][WFO_CHAMP[ticker]]["total"],
                            wfo_sharpe=WFO_METRICS[ticker][WFO_CHAMP[ticker]]["sharpe"]))

al_count = sum(1 for s in all_signals if "AL" in s["sinyal"])
print(f"\n{SEP}")
print(f"  ÖZET: {al_count}/{len(HISSELER)} hisse AL sinyali veriyor")
print(f"  Toplam pozisyon: {sum(s['pozisyon_tl'] for s in all_signals if 'AL' in s['sinyal']):,.0f} TL")
print(SEP)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("BIST Gerçek Veri — Portföy Analizi  (2026-06-06)", fontsize=13, fontweight="bold")

# Panel 1: WFO total returns bar chart
ax1 = axes[0]
xs  = HISSELER
ys  = [WFO_METRICS[t][WFO_CHAMP[t]]["total"] for t in xs]
bhs = [WFO_METRICS[t]["BuyHold"]["total"] for t in xs]
x   = np.arange(len(xs)); w = 0.35
b1 = ax1.bar(x - w/2, ys,  w, label="WFO Şampiyon", color="#2196F3", alpha=0.85)
b2 = ax1.bar(x + w/2, bhs, w, label="Buy & Hold",   color="#9E9E9E", alpha=0.7)
ax1.axhline(0, color="white", lw=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(xs)
ax1.set_ylabel("Toplam Getiri (%)"); ax1.set_title("WFO vs Buy-Hold Getiri")
ax1.legend()
for bar in b1: ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                         f"{bar.get_height():.0f}%", ha="center", fontsize=8)

# Panel 2: Sharpe comparison
ax2 = axes[1]
sharpes = {s: [WFO_METRICS[t].get(s,{}).get("sharpe",0) for t in HISSELER]
           for s in ["MACT","MOM3","RAM","HMM","ENSE"]}
x2 = np.arange(len(HISSELER))
bar_w = 0.15
clrs2 = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336"]
for i,(sn,svals) in enumerate(sharpes.items()):
    ax2.bar(x2 + i*bar_w - 0.3, svals, bar_w, label=sn, color=clrs2[i], alpha=0.8)
ax2.set_xticks(x2); ax2.set_xticklabels(HISSELER)
ax2.axhline(0, color="white", lw=0.5)
ax2.set_ylabel("Sharpe Oranı"); ax2.set_title("Strateji Sharpe Oranları")
ax2.legend(fontsize=8)

# Panel 3: Risk-Return scatter
ax3 = axes[2]
for t in HISSELER:
    m   = WFO_METRICS[t][WFO_CHAMP[t]]
    k   = KELLY_INFO[t]
    col = "#F44336" if "AL" in [s["sinyal"] for s in all_signals if s["ticker"]==t] else "#607D8B"
    ax3.scatter(abs(m["maxdd"]), m["total"], s=k["kelly"]*2000+50,
                color=col, alpha=0.8, zorder=5)
    ax3.annotate(t, (abs(m["maxdd"]), m["total"]), textcoords="offset points",
                 xytext=(5,5), fontsize=9)
ax3.axhline(0, color="white", lw=0.5)
ax3.set_xlabel("Max Drawdown (abs %)"); ax3.set_ylabel("Toplam Getiri (%)")
ax3.set_title("Risk-Getiri  (boyut = Kelly%)")
from matplotlib.patches import Patch
ax3.legend(handles=[Patch(color="#F44336",label="AL Sinyali"),
                     Patch(color="#607D8B",label="FLAT")], loc="best")

plt.tight_layout()
plt.savefig("/tmp/portfoy_analiz.png", dpi=120, bbox_inches="tight")
plt.show()
print("Grafik: /tmp/portfoy_analiz.png")
